# Recommendation System Assignment_ 16

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Tasks:

### Data Preprocessing:

#### Load the dataset into a suitable data structure (e.g., pandas DataFrame).
#### Handle missing values, if any.
#### Explore the dataset to understand its structure and attributes

In [46]:
#Load the dataset
df=pd.read_csv('anime.csv')
#Display data
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [55]:
#Dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12294 non-null  object 
 3   type      12294 non-null  object 
 4   episodes  12294 non-null  float64
 5   rating    12294 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(2), int64(2), object(3)
memory usage: 672.5+ KB


In [49]:
# Convert episodes if needed (safe step)
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

In [56]:
#Dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12294 non-null  object 
 3   type      12294 non-null  object 
 4   episodes  12294 non-null  float64
 5   rating    12294 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(2), int64(2), object(3)
memory usage: 672.5+ KB


In [47]:
#Check missing value
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [54]:
#check mode value
type_df=df['type'].mode()[0]
type_df

'TV'

In [51]:
#check mean value()
rating=df['rating'].mean()
np.round(rating,2)

np.float64(6.47)

In [52]:
#Handle missing
df.fillna({
    'genre':'Unknown',
    'type':type_df,
    'rating':rating
},inplace=True)

In [41]:
#again check missing value
df.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

In [42]:
#shape
df.shape

(12294, 7)

In [43]:
#number of #unique name's
df['name'].nunique()

12292

In [44]:
#number of #unique name's
df['anime_id'].nunique()

12294

In [57]:
#drop duplicates and reset index
df = df.drop_duplicates(subset='name')
df.reset_index(drop=True, inplace=True)

In [58]:
#number of #unique name's
df['name'].nunique()

12292

In [59]:
#number of #unique name's
df['anime_id'].nunique()

12292

### Feature Extraction:

#### Convert categorical features into numerical representations if necessary.


In [60]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack

In [62]:
# Convert Genre → Numerical

vectorizer = CountVectorizer()
genre_matrix = vectorizer.fit_transform(df['genre'].str.replace(", ", " "))

#### Normalize numerical features if required.

In [63]:
# Normalize Numerical Features

scaler = MinMaxScaler()
numeric_features = scaler.fit_transform(df[['rating', 'episodes', 'members']])

In [65]:
# Combine Features
combined_features = hstack([genre_matrix, numeric_features])

#### Decide on the features that will be used for computing similarity (e.g., genres, user ratings).

### Recommendation System:

#### Design a function to recommend anime based on cosine similarity.

In [66]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute similarity
cosine_sim = cosine_similarity(combined_features)

In [67]:
# Recommendation Function

def recommend_anime(title, top_n=5, threshold=0.4):
    
    if title not in df['name'].values:
        return "Anime not found"
    
    idx = df[df['name'] == title].index[0]
    
    # Similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Filter threshold & remove itself
    sim_scores = [x for x in sim_scores if x[1] >= threshold and x[0] != idx]
    
    # Top N
    sim_scores = sim_scores[:top_n]
    
    anime_indices = [i[0] for i in sim_scores]
    
    return df[['name', 'genre']].iloc[anime_indices]

#### Given a target anime, recommend a list of similar anime based on cosine similarity scores.


In [68]:
# Example
recommend_anime("Naruto")

,name,genre
615,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P..."
1472,Naruto: Shippuuden Movie 4 - The Lost Tower,"Action, Comedy, Martial Arts, Shounen, Super P..."
1573,Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...,"Action, Comedy, Martial Arts, Shounen, Super P..."
486,Boruto: Naruto the Movie,"Action, Comedy, Martial Arts, Shounen, Super P..."
1343,Naruto x UT,"Action, Comedy, Martial Arts, Shounen, Super P..."


#### Experiment with different threshold values for similarity scores to adjust the recommendation list size.


In [69]:
# Low threshold → more recommendations
recommend_anime("Naruto", threshold=0.3)

# Medium threshold
recommend_anime("Naruto", threshold=0.5)

# High threshold → fewer but accurate
recommend_anime("Naruto", threshold=0.7)

,name,genre
615,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P..."
1472,Naruto: Shippuuden Movie 4 - The Lost Tower,"Action, Comedy, Martial Arts, Shounen, Super P..."
1573,Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...,"Action, Comedy, Martial Arts, Shounen, Super P..."
486,Boruto: Naruto the Movie,"Action, Comedy, Martial Arts, Shounen, Super P..."
1343,Naruto x UT,"Action, Comedy, Martial Arts, Shounen, Super P..."


#### Analyze the performance of the recommendation system and identify areas of improvement.

### Interview Questions:
#### 1. Can you explain the difference between user-based and item-based collaborative filtering?

#### 2. What is collaborative filtering, and how does it work?